In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from utils.constants import BATCH_SIZE, NEAREST_NEIGHBOUR_COUNT
from tqdm import tqdm
from pathlib import Path
from chunking import PythonChunker
from code_retriever import FAISS, HybridSearch, BM25_Plus, ReciprocalRankFusion, SearchResult, DocumentStore
from evaluator import Evaluator, SearchQueryData
from utils.embed_input import datadict2embed_input, jsonl2datadict
from pprint import pprint
import json
from chunking import PythonChunker

In [7]:
doc_store = DocumentStore()
dense = FAISS(batch_size=BATCH_SIZE)
bm25 = BM25_Plus()
py_chunker = PythonChunker()

curr_dir = Path.cwd().parent / "data" / "test-repo"
print(curr_dir)
with open("../output.jsonl", "w") as f:
    py_chunker.chunk_files(curr_dir, f)

with open("../output.jsonl", "r") as f:
    lines = f.readlines()
    n_chunks = len(lines)
    metadatas = list(map(jsonl2datadict, lines))

target: list[SearchQueryData] = []
with open("../rag_eval_dataset_natural_language.jsonl", "r") as f:
    for line in f.readlines():
        target.append(json.loads(line))


texts: list[str] = []
chunk_ids: list[str] = []
for i in range(n_chunks):
    embed_input = datadict2embed_input(metadatas[i])
    chunk_ids.append(metadatas[i].chunk_id)
    doc_store.insert(metadatas[i])
    texts.append(embed_input)

hybrid_search = HybridSearch([dense, bm25], ReciprocalRankFusion())
hybrid_search.fit(texts, chunk_ids)

/home/gaurav/coding/SoftwareEngineeringAgent/data/test-repo
Embedding and saving batches: 


100%|██████████| 95/95 [00:03<00:00, 27.92it/s]


In [8]:
q = ["Helper method to get MongoDB collection for getting books"]

hybrid_result = hybrid_search.search(q, 180)
ids = [res.chunk_id for res in hybrid_result[0]]
res = doc_store.get(ids)
pprint(res)

[ChunkMetaData(chunk_id='32830686-0fbe-4671-8576-ed63ea2e6087',
               chunk_type='method',
               name='_get_collection',
               qualified_name='module.BookController._get_collection',
               parent_name='BookController',
               parent_type='class',
               file_path='backend/src/controllers/book_controller.py',
               file_name='book_controller.py',
               module_path='backend.src.controllers.book_controller',
               language='Python',
               start_line=27,
               end_line=32,
               source_code='def _get_collection(self, collection_name: str):\n'
                           '        """\n'
                           '        Helper method to get MongoDB collection.\n'
                           '        """\n'
                           '        # '
                           'self.mongo_db_connection.start_connection()\n'
                           '        return '
                       

In [11]:
e = Evaluator()
e.evaluate(10, lambda x, y: doc_store.get_from_search_result(hybrid_search.search(x, y)), target)

{'precision': 0.2710000000000001,
 'recall': 0.6226666666666665,
 'mrr': 0.7371349206349204}